---
jupyter: false
---

# Extracting Hand-Motion Features

This notebook turns each standardized video into time-series features that can be used by the LSTM model.

The basic computer vision problem is: how do we turn a video of a hand into numbers? Here we use a simple mask-based approach to estimate hand position and motion over time.


## Settings

The settings below control visualization and the mask-building pipeline. Most students should read these as parameters rather than as values to memorize.


### Visualization Options

These flags control what OpenCV windows appear while the notebook runs. They are useful for debugging because mask-based tracking can fail if the lighting, skin threshold, or background subtraction is poor.


In [ ]:
# Visualization options
DO_NOT_SHOW_MASKS = False
DEBUG_GRID = True

SHOW_RAW_FRAMES = False
SHOW_SKIN_MASK = False
SHOW_FOREGROUND_MASK = False
SHOW_COMBINED_MASK = False
SHOW_BAD_MASKS = False
SHOW_HAND_CONTOURS = False

FRAME_DELAY_MS = 100  # Adjust to control playback speed
# Set this above 0 only when intentionally resuming a partial run.
START_AT_VIDEO_IDX = 0


### Mask Parameters

The feature extractor uses two kinds of evidence:

- **Foreground motion:** pixels that changed relative to the learned background.
- **Skin-like color:** pixels whose color falls within a rough skin-color range.

The masks are cleaned with morphological opening and closing. This removes small specks and fills small holes before tracking the largest connected region.


In [ ]:

# Background subtraction (motion)
BG_HISTORY = 200
BG_VAR_THRESHOLD = 16
BG_DETECT_SHADOWS = True
BG_WARMUP_FRAMES = 30
BG_FREEZE_AFTER_WARMUP = True
BG_LEARNING_RATE = -1.0

# Color segmentation (skin-like)
SKIN_YCRCB_LOWER = (0, 135, 85)
SKIN_YCRCB_UPPER = (255, 180, 135)
MIN_SAT = 30

# Mask cleanup
MORPH_KERNEL = (5, 5)
MORPH_OPEN_ITERS = 1
MORPH_CLOSE_ITERS = 2
MIN_AREA_CUTOFF = 20  # Minimum area to consider a contour as valid object

In [ ]:
# Imports and dataset location
import cv2
import polars as pl
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import LoadVideo as lv
import videofeatures as feat
from MotionEnergy import moeng_video
DATA_ROOT = lv.get_data_root()

## Build Masks and Track the Hand

This section defines the core computer vision functions.

The pipeline for each frame is:

1. Build a foreground-motion mask.
2. Build a skin-color mask.
3. Combine and clean masks.
4. Find the largest valid region.
5. Use the region centroid as the hand position.

This is a simple approach, but it is useful for teaching because each step is visible and interpretable.


In [ ]:
BG_SUBTRACTOR = cv2.createBackgroundSubtractorMOG2(
    history=BG_HISTORY,
    varThreshold=BG_VAR_THRESHOLD,
    detectShadows=BG_DETECT_SHADOWS,
)

def show_frame(frame: np.ndarray, title: str = "Frame", show: bool = True):
    if not show:
        return
    cv2.imshow(title, frame)
    key = cv2.waitKey(FRAME_DELAY_MS) & 0xFF
    if key == ord("q"):
        cv2.destroyAllWindows()
        raise SystemExit("Stopped by user.")

def clean_mask(mask: np.ndarray) -> np.ndarray:
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, MORPH_KERNEL)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=MORPH_OPEN_ITERS)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=MORPH_CLOSE_ITERS)
    return mask

def _to_bgr(frame: np.ndarray) -> np.ndarray:
    if frame.ndim == 2:
        return cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
    return frame

def build_debug_grid(color_frame: np.ndarray, fg_mask: np.ndarray, skin_mask: np.ndarray, combined: np.ndarray, overlay: np.ndarray):
    tiles = []
    tiles.append(_to_bgr(color_frame))
    tiles.append(_to_bgr(fg_mask))
    tiles.append(_to_bgr(skin_mask))
    tiles.append(_to_bgr(combined))
    tiles.append(_to_bgr(overlay))

    h, w = tiles[0].shape[:2]
    tiles = [cv2.resize(t, (w, h), interpolation=cv2.INTER_NEAREST) for t in tiles]
    blank = np.zeros_like(tiles[0])
    row1 = np.hstack([tiles[0], tiles[1]])
    row2 = np.hstack([tiles[2], tiles[3]])
    row3 = np.hstack([tiles[4], blank])
    grid = np.vstack([row1, row2, row3])
    return grid

# Background subtraction finds pixels that changed relative to the scene background.
def build_combined_mask(frame: np.ndarray, frame_idx: int):
    if frame.ndim == 2:
        color_frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
    else:
        color_frame = frame

    learning_rate = BG_LEARNING_RATE
    if BG_FREEZE_AFTER_WARMUP and frame_idx >= BG_WARMUP_FRAMES:
        learning_rate = 0

    fg_mask = BG_SUBTRACTOR.apply(color_frame, learningRate=learning_rate)
    fg_mask = (fg_mask == 255).astype(np.uint8) * 255

    ycrcb = cv2.cvtColor(color_frame, cv2.COLOR_BGR2YCrCb)
    skin_mask = cv2.inRange(ycrcb, SKIN_YCRCB_LOWER, SKIN_YCRCB_UPPER)
    if MIN_SAT > 0:
        hsv = cv2.cvtColor(color_frame, cv2.COLOR_BGR2HSV)
        sat_mask = (hsv[:, :, 1] > MIN_SAT).astype(np.uint8) * 255
        skin_mask = cv2.bitwise_and(skin_mask, sat_mask)

    fg_mask = clean_mask(fg_mask)
    skin_mask = clean_mask(skin_mask)
    combined = cv2.bitwise_and(fg_mask, skin_mask)
    combined = clean_mask(combined)
    return combined, fg_mask, skin_mask, color_frame

# Use image moments to convert the largest mask region into an x/y centroid.
def find_mask_centroid(mask: np.ndarray):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None

    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < MIN_AREA_CUTOFF:
        return None, None

    M = cv2.moments(cnt)
    if M["m00"] == 0:
        return None, None

    cx = int(M["m10"] / M["m00"])
    cy = int(M["m01"] / M["m00"])
    return (cx, cy), cnt


## Process Frames and Videos

The runner functions apply the mask pipeline to every frame in a video. For each video, the notebook saves:

- `features_position.csv`: x/y hand position by frame.
- `features_moeng_skin.csv`: motion energy from the skin mask.
- `features_moeng_motion.csv`: motion energy from the foreground-motion mask.
- `features_moeng_combined.csv`: motion energy from the combined mask.

These files are the bridge between computer vision and the modeling notebook.


In [ ]:

def process_frame(frame: np.ndarray, frame_idx: int, show_masks: bool = not DO_NOT_SHOW_MASKS):
    combined, fg_mask, skin_mask, color_frame = build_combined_mask(frame, frame_idx)

    centroid, cnt = find_mask_centroid(combined)
    if centroid is None:
        if show_masks:
            if DEBUG_GRID:
                overlay = color_frame.copy()

                grid = build_debug_grid(color_frame, fg_mask, skin_mask, combined, overlay)


                show_frame(grid, title="Debug Grid", show=True)
            else:
                show_frame(combined, title="Mask (no contours)", show=SHOW_BAD_MASKS)
        return None, combined, fg_mask, skin_mask

    overlay = color_frame.copy()
    cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 1)
    cv2.circle(overlay, centroid, 3, (0, 0, 255), -1)
    if show_masks:
        if DEBUG_GRID:
            # if frame_idx < 5:
                # print(frame_idx)
                # key = cv2.waitKey(2000) & 0xFF            
            grid = build_debug_grid(color_frame, fg_mask, skin_mask, combined, overlay)
            show_frame(grid, title="Debug Grid", show=True)
        else:
            show_frame(color_frame, title="Frame", show=SHOW_RAW_FRAMES)
            show_frame(fg_mask, title="Mask (motion)", show=SHOW_FOREGROUND_MASK)
            show_frame(skin_mask, title="Mask (skin)", show=SHOW_SKIN_MASK)
            show_frame(combined, title="Mask (combined)", show=SHOW_COMBINED_MASK)
            show_frame(overlay, title="Contours (combined)", show=SHOW_HAND_CONTOURS)

    return centroid, combined, fg_mask, skin_mask


In [ ]:
def process_video(video_id:str, plot=False):

    vid_full_path = lv.get_video_file(video_id, kind="standardized", video_root=DATA_ROOT, require_exists=True)
    print(f"Video ID: {video_id}")
    print(f"Full video path:\n {vid_full_path}")
    video_np_array = lv.get_np_video_array(vid_full_path,verbose=False,plot_eg_frame=False)

    rows = []
    # Store masks so we can compute motion energy on each mask stream.
    skin_masks = []
    motion_masks = []
    combined_masks = []

    for i, frame in enumerate(video_np_array):
        pos, combined, motion_mask, skin_mask = process_frame(frame, i)
        rows.append({
            "FrameIndex": i,
            "x": None if pos is None else pos[0],
            "y": None if pos is None else pos[1],
        })
        skin_masks.append(skin_mask)
        motion_masks.append(motion_mask)
        combined_masks.append(combined)

    cv2.destroyAllWindows()
    df_pos = pl.DataFrame(rows)
    print(df_pos.head())

    # plot position over time
    if plot:
        pandas_df = df_pos.to_pandas()
        print(pandas_df.head())
        plt.figure(figsize=(12, 6))
        sns.lineplot(data=pandas_df, x="FrameIndex", y="x", label="X Position")
        sns.lineplot(data=pandas_df, x="FrameIndex", y="y", label="Y Position")
        plt.title('Object Position Over Time')
        plt.xlabel('Frame Index')
        plt.ylabel('Position (pixels)')
        plt.legend()
        plt.show()

    # Save to CSV
    output_csv_path = feat.vid_extracted_feature_path(video_id, "position", video_root=DATA_ROOT, require_path_exists=False)
    df_pos.write_csv(output_csv_path)

    df_skin_me = moeng_video(np.asarray(skin_masks), force_gray=False)
    df_motion_me = moeng_video(np.asarray(motion_masks), force_gray=False)
    df_combined_me = moeng_video(np.asarray(combined_masks), force_gray=False)

    df_skin_me_save = df_skin_me.rename({"MotionEnergy": "moeng_skin"})
    df_motion_me_save = df_motion_me.rename({"MotionEnergy": "moeng_motion"})
    df_combined_me_save = df_combined_me.rename({"MotionEnergy": "moeng_combined"})

    df_skin_me_save.write_csv(feat.vid_extracted_feature_path(video_id, "moeng_skin", video_root=DATA_ROOT, require_path_exists=False))
    df_motion_me_save.write_csv(feat.vid_extracted_feature_path(video_id, "moeng_motion", video_root=DATA_ROOT, require_path_exists=False))
    df_combined_me_save.write_csv(feat.vid_extracted_feature_path(video_id, "moeng_combined", video_root=DATA_ROOT, require_path_exists=False))

    if plot:
        df_me = pl.concat([
            df_skin_me.with_columns(pl.lit("skin").alias("MaskType")),
            df_motion_me.with_columns(pl.lit("motion").alias("MaskType")),
            df_combined_me.with_columns(pl.lit("combined").alias("MaskType")),
        ])

        pandas_me_df = df_me.to_pandas()
        plt.figure(figsize=(12, 6))
        sns.lineplot(data=pandas_me_df, x="FrameIndex", y="MotionEnergy", hue="MaskType")
        plt.title('Motion Energy Over Time')
        plt.xlabel('Frame Index')
        plt.ylabel('Motion Energy Value')
        plt.legend()
        plt.show()


In [ ]:

def main():
    from tqdm import tqdm

    VIDEO_IDS = lv.list_video_ids(DATA_ROOT)
    VIDEO_IDS = VIDEO_IDS[START_AT_VIDEO_IDX:]
    assert len(VIDEO_IDS) > 0, f"No video folders found in: {DATA_ROOT}"

    rows = []
    for video_id in tqdm(VIDEO_IDS, desc="Extracting Features"):
        lv.get_video_dir(video_id, video_root=DATA_ROOT, require_exist=True)
        process_video(video_id, plot=False)

if __name__ == "__main__":
    main()        


## Takeaway

This notebook converts video into model-ready time series. The extracted x/y positions describe where the hand is, while speed, direction, and motion-energy features can be derived from those frame-by-frame measurements.

The next notebook uses these features as inputs to an LSTM model.